# LightGBM classifier 
using native categorical feature support(pandas "category" dtype, passed via categorical_feature).

In [1]:
import os
import sys
import importlib
import scipy
import sklearn
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

print(np.__version__)
print(pd.__version__)
print(lgb.__version__)

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

fe = importlib.import_module("06_Feature_Engineering")
cv = importlib.import_module("07_Cross_Validation")

MODEL_NAME = "lightgbm"
ARTIFACT_DIR = "./artifacts"
MODEL_DIR = "./models"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

LGB_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_samples=30,
    objective="binary",
    metric="auc",
    random_state=cv.SEED,
    n_jobs=-1,
    verbose=-1,
)

1.26.4
2.3.3
4.3.0


In [2]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    train, test = fe.load_raw_data()
    train = fe.engineer_features(train)
    test = fe.engineer_features(test)
    num_cols, cat_cols = fe.get_feature_lists(train)
    feature_cols = num_cols + cat_cols

    for c in cat_cols:
        train[c] = train[c].astype("category")
        test[c] = test[c].astype("category")

    y = train[fe.TARGET].values
    fold_ids = cv.get_or_create_folds(train, target_col=fe.TARGET)

    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []
    importances = np.zeros(len(feature_cols))

    print("=" * 70)
    print(f"LIGHTGBM ({cv.N_SPLITS}-fold CV)")
    print("=" * 70)

    for fold in range(cv.N_SPLITS):
        train_idx, valid_idx = cv.fold_split(train, fold_ids, fold)

        X_train, y_train = train.loc[train_idx, feature_cols], y[train_idx]
        X_valid, y_valid = train.loc[valid_idx, feature_cols], y[valid_idx]

        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(200)]
        )

        valid_pred = model.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred

        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f} (best_iter={model.best_iteration_})")

        test_pred += model.predict_proba(test[feature_cols])[:, 1] / cv.N_SPLITS
        importances += model.feature_importances_ / cv.N_SPLITS
        model.booster_.save_model(f"{MODEL_DIR}/lightgbm_fold{fold}.txt")

    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    cv.summarize_oof(y, oof_pred, MODEL_NAME)

    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("\nTop feature importances:")
    print(imp_df.head(15).to_string(index=False))

    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({fe.ID_COL: test[fe.ID_COL], fe.TARGET: test_pred}).to_csv(
        SUB_PATH, index=False
    )
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")

In [3]:
if __name__ == "__main__":
    main()

LIGHTGBM (5-fold CV)
[200]	valid_0's auc: 0.940171
[400]	valid_0's auc: 0.940281
Fold 0: AUC = 0.94032 (best_iter=449)
[200]	valid_0's auc: 0.941219
[400]	valid_0's auc: 0.94128
Fold 1: AUC = 0.94128 (best_iter=407)
[200]	valid_0's auc: 0.942579
[400]	valid_0's auc: 0.942618
Fold 2: AUC = 0.94265 (best_iter=340)
[200]	valid_0's auc: 0.942107
[400]	valid_0's auc: 0.942173
Fold 3: AUC = 0.94218 (best_iter=420)
[200]	valid_0's auc: 0.941363
[400]	valid_0's auc: 0.94147
Fold 4: AUC = 0.94149 (best_iter=412)

Mean fold AUC: 0.94158 (+/- 0.00080)
[lightgbm] OOF ROC-AUC: 0.94157

Top feature importances:
                      feature  importance
            Annual_Income_USD      5070.8
             Daily_Commute_km      3155.0
               Income_per_Age      2769.2
              Commute_per_Car      2341.2
                          Age      2227.2
       Charging_Station_Ratio      1853.4
      Total_Charging_Stations      1122.0
  Charging_Stations_Near_Work      1090.4
        Charging_